<a href="https://colab.research.google.com/github/CodeHunterOfficial/ABCD_ASPNETCORE/blob/main/TatarNLP/kiziltan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================
# 📦 Установка зависимостей
# =====================================================
# !pip install requests beautifulsoup4 python-dateutil tqdm > /dev/null

# =====================================================
# 🧰 Импорт и настройки
# =====================================================
import json
import time
import hashlib
import requests
import re
import random
import os
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from dateutil import parser as dateparser
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

CONTACT_EMAIL = "researcher@example.com"
USER_AGENT = f"KizilTanScraper/1.0 (+{CONTACT_EMAIL})"
HEADERS = {
    "User-Agent": USER_AGENT,
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "tt,ru;q=0.9,en;q=0.8",
}
DEFAULT_DELAY = 1.5
RUBRIC_DELAY = (1.5, 3.0)
EXCERPT_LEN = 200
BASE_URL = "https://kiziltan.ru/"
MAX_THREADS = 5
OUTPUT_FILE = "kiziltan_articles.jsonl"

# =====================================================
# 🔍 Вспомогательные функции
# =====================================================
def get_domain(url):
    return urlparse(url).netloc

def clean_text(text):
    if not text:
        return ""
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def fetch_requests(url, timeout=20):
    try:
        time.sleep(random.uniform(0.5, 1.2))
        r = requests.get(url, headers=HEADERS, timeout=timeout)
        r.raise_for_status()
        return r.text
    except Exception as e:
        print(f"❌ Ошибка загрузки {url}: {e}")
        return None

def check_url_exists(url):
    """Быстрая проверка существования URL"""
    try:
        response = requests.head(url, headers=HEADERS, timeout=5, allow_redirects=True)
        return response.status_code != 404
    except:
        return True  # Если не можем проверить, предполагаем что существует

# =====================================================
# 🌐 Сбор ссылок всех рубрик (адаптировано для kiziltan.ru)
# =====================================================
def collect_rubric_links():
    print("🔍 Загружаем главную страницу...")
    html = fetch_requests(BASE_URL)
    if not html:
        return []

    soup = BeautifulSoup(html, "html.parser")
    rubrics = set()

    # Находим все ссылки на рубрики из навигационного меню
    # Ищем в меню "Разделы" и "Рубрикалар"
    menu_sections = soup.find_all(['div', 'ul'], class_=re.compile(r'menu|rubrics|sections|nav'))

    for section in menu_sections:
        for link in section.find_all('a', href=True):
            href = link['href']
            text = link.get_text(strip=True)

            # Фильтруем только ссылки на статьи и рубрики
            if '/articles/' in href and len(text) < 50 and text:
                full_url = urljoin(BASE_URL, href)
                rubrics.add((full_url, text))

    # Если не нашли, ищем все ссылки с /articles/ на главной
    if not rubrics:
        for link in soup.find_all('a', href=True):
            href = link['href']
            if '/articles/' in href and not any(x in href for x in ['2026', '2025', '2024']):
                text = link.get_text(strip=True)
                if text and len(text) < 50:
                    full_url = urljoin(BASE_URL, href)
                    rubrics.add((full_url, text))

    # Добавляем основные рубрики из структуры сайта
    known_rubrics = [
        ("Милләт", "/articles/millt"),
        ("Батырлар дәвере", "/articles/batyrlar-d-vere"),
        ("Яңалыклар", "/articles/novosti"),
        ("Нацпроекты", "/articles/nacproekty"),
        ("Мәгариф", "/articles/obraz"),
        ("Мәдәният", "/articles/kultura"),
        ("Республика", "/articles/republic"),
        ("Икътисад", "/articles/ekonomika"),
        ("Гаилә учагы", "/articles/gail-uchagy"),
        ("Җәмгыять", "/articles/obchestvo"),
        ("Сәламәтлек", "/articles/zdorovie"),
        ("Сәясәт", "/articles/politika"),
        ("Спорт", "/articles/sport"),
        ("Дөнья", "/articles/donja"),
        ("Әдәбият", "/articles/d-biyat"),
        ("Бөек Җиңүгә - 80 ел", "/pobeda-80"),
    ]

    for name, path in known_rubrics:
        full_url = urljoin(BASE_URL, path)
        rubrics.add((full_url, name))

    rubric_list = [{'url': url, 'name': name} for url, name in rubrics]
    # Сортируем по названию
    rubric_list.sort(key=lambda x: x['name'])

    print(f"📊 Найдено рубрик: {len(rubric_list)}")

    # Выводим первые 10 рубрик
    for r in rubric_list[:10]:
        print(f"  • {r['name']}: {r['url']}")
    if len(rubric_list) > 10:
        print(f"  ... и ещё {len(rubric_list) - 10} рубрик")

    return rubric_list

# =====================================================
# 📰 Сбор ссылок статей из рубрики (адаптировано для kiziltan.ru)
# =====================================================
def collect_articles_from_rubric(rubric):
    rubric_url = rubric['url']
    rubric_name = rubric['name']

    print(f"📂 Рубрика: {rubric_name}")
    article_urls = set()
    page = 1
    max_pages = 30  # Максимальное количество страниц

    while page <= max_pages:
        # Формируем URL страницы
        if page == 1:
            page_url = rubric_url
        else:
            page_url = f"{rubric_url.rstrip('/')}/page/{page}"

        # Проверяем существование страницы
        if page > 1 and not check_url_exists(page_url):
            print(f"  ⏹ Страница {page} не существует")
            break

        print(f"  📄 Страница {page}: {page_url}")
        html = fetch_requests(page_url)
        if not html:
            break

        soup = BeautifulSoup(html, "html.parser")

        # Ищем ссылки на статьи
        article_links = set()

        # Способ 1: Ищем в контейнере со статьями
        items_container = soup.find('div', class_='page-container')
        if items_container:
            for link in items_container.find_all('a', href=True):
                href = link['href']
                if '/articles/' in href and re.search(r'/2026/\d{2}/\d{2}/', href):
                    full_url = urljoin(BASE_URL, href)
                    article_links.add(full_url)

        # Способ 2: Ищем все ссылки с датами в URL
        if not article_links:
            for link in soup.find_all('a', href=True):
                href = link['href']
                # Ищем ссылки вида /articles/.../2026-02-21/...
                if '/articles/' in href and re.search(r'/2026-\d{2}-\d{2}/', href):
                    full_url = urljoin(BASE_URL, href)
                    article_links.add(full_url)

        # Способ 3: Ищем по паттерну URL статьи
        if not article_links:
            for link in soup.find_all('a', href=True):
                href = link['href']
                if '/articles/' in href and len(href.split('/')) > 4:
                    if not any(x in href for x in ['page', 'rubric', 'tag']):
                        full_url = urljoin(BASE_URL, href)
                        article_links.add(full_url)

        # Добавляем только новые ссылки
        new_articles = [url for url in article_links if url not in article_urls]
        article_urls.update(new_articles)

        print(f"  → Найдено новых статей: {len(new_articles)}")

        if len(new_articles) == 0:
            print("  ⚠ Новых статей не найдено")
            break

        # Проверяем наличие следующей страницы
        next_link = None

        # Ищем по тексту
        next_text = soup.find('a', string=re.compile(r'Следующая|Киләсе|→|›|»|Next'))
        if next_text and next_text.get('href'):
            next_link = next_text

        # Ищем в пагинации
        if not next_link:
            pagination = soup.find('ul', class_='pagination')
            if pagination:
                next_page = pagination.find('a', href=re.compile(rf'/page/{page+1}'))
                if next_page:
                    next_link = next_page

        if not next_link:
            print(f"  ⏹ Достигнут конец пагинации")
            break

        page += 1
        time.sleep(DEFAULT_DELAY)

    print(f"  ✅ Всего собрано: {len(article_urls)} статей\n")
    return list(article_urls)

# =====================================================
# 🧩 Извлечение данных со страницы статьи (адаптировано для kiziltan.ru)
# =====================================================
def extract_author(soup):
    # Ищем автора в мета-тегах
    meta = soup.find('meta', attrs={'name': 'author'})
    if meta and meta.get('content'):
        return meta['content'].strip()

    # Ищем автора в JSON-LD
    script = soup.find('script', type='application/ld+json')
    if script:
        try:
            data = json.loads(script.string)
            if isinstance(data, dict) and 'author' in data:
                if isinstance(data['author'], dict) and 'name' in data['author']:
                    return data['author']['name']
                elif isinstance(data['author'], list) and len(data['author']) > 0:
                    return data['author'][0].get('name', '')
        except:
            pass

    # Ищем автора в тексте статьи
    author_elem = soup.find('span', class_='author') or soup.find('div', class_='author')
    if author_elem:
        return clean_text(author_elem.get_text())

    return "Кызыл таң"

def extract_date(soup):
    # Ищем дату в мета-тегах
    meta = soup.find('meta', attrs={'property': 'article:published_time'})
    if meta and meta.get('content'):
        try:
            return dateparser.parse(meta['content']).isoformat()
        except:
            return meta['content']

    # Ищем дату в JSON-LD
    script = soup.find('script', type='application/ld+json')
    if script:
        try:
            data = json.loads(script.string)
            if isinstance(data, dict) and 'datePublished' in data:
                return dateparser.parse(data['datePublished']).isoformat()
        except:
            pass

    # Ищем дату в тексте
    date_elem = soup.find('time', class_='date') or soup.find('span', class_='date')
    if date_elem:
        date_text = clean_text(date_elem.get_text())
        try:
            return dateparser.parse(date_text, dayfirst=True).isoformat()
        except:
            return date_text

    return None

def extract_category(soup, url):
    # Извлекаем категорию из URL
    match = re.search(r'/articles/([^/]+)/', url)
    if match:
        cat_code = match.group(1)
        # Словарь соответствий для известных рубрик
        cat_names = {
            'millt': 'Милләт',
            'batyrlar-d-vere': 'Батырлар дәвере',
            'novosti': 'Яңалыклар',
            'nacproekty': 'Нацпроекты',
            'obraz': 'Мәгариф',
            'kultura': 'Мәдәният',
            'republic': 'Республика',
            'ekonomika': 'Икътисад',
            'gail-uchagy': 'Гаилә учагы',
            'obchestvo': 'Җәмгыять',
            'zdorovie': 'Сәламәтлек',
            'politika': 'Сәясәт',
            'sport': 'Спорт',
            'donja': 'Дөнья',
            'd-biyat': 'Әдәбият',
            'pobeda-80': 'Бөек Җиңүгә - 80 ел',
        }
        return cat_names.get(cat_code, cat_code.replace('-', ' ').title())

    # Пробуем найти в хлебных крошках
    breadcrumbs = soup.find('ul', class_='breadcrumb')
    if breadcrumbs:
        links = breadcrumbs.find_all('a')
        if len(links) >= 2:
            return clean_text(links[1].get_text())

    return "Гомум яңалыклар"  # Категория по умолчанию

def extract_content(soup):
    """Извлечение основного текста статьи с учётом структуры kiziltan.ru"""
    content_blocks = []

    # Основной контейнер статьи (по структуре kiziltan.ru)
    content_area = (
        soup.find('div', class_='paragraph serif-text') or   # прямой контейнер текста
        soup.find('div', class_='block_1h4f8') or            # внешний блок
        soup.find('div', class_=re.compile(r'paragraph'))    # запасной вариант
    )

    if not content_area:
        # Если ничего не нашли – пробуем старые селекторы
        content_area = (soup.find('div', itemprop='articleBody') or
                       soup.find('article') or
                       soup.find('main'))

    if content_area:
        # Удаляем ненужные элементы
        for elem in content_area.find_all(['script', 'style', 'noscript', 'iframe']):
            elem.decompose()

        # Удаляем рекламные блоки и виджеты соцсетей
        for elem in content_area.find_all(['div', 'aside']):
            if elem.get('class'):
                class_str = ' '.join(elem.get('class')).lower()
                if any(x in class_str for x in ['ad', 'adv', 'banner', 'social', 'share', 'telegram', 'vk', 'facebook']):
                    elem.decompose()

        # Извлекаем текст из параграфов и заголовков
        for elem in content_area.find_all(['p', 'h2', 'h3', 'h4']):
            text = clean_text(elem.get_text())
            if len(text) > 20:   # фильтруем слишком короткие фрагменты
                if elem.name in ['h2', 'h3', 'h4']:
                    content_blocks.append(f"\n{text}\n")
                else:
                    content_blocks.append(text)

    return "\n\n".join(content_blocks).strip()

def extract_lead(soup):
    """Извлечение лида (краткого описания)"""
    # Ищем по классу bigLead_37UJd (из структуры страницы)
    lead_elem = soup.find('h2', class_='bigLead_37UJd')
    if lead_elem:
        return clean_text(lead_elem.get_text())

    # Резервные варианты
    lead_elem = soup.find('div', class_='lead') or soup.find('p', class_='lead')
    if lead_elem:
        return clean_text(lead_elem.get_text())

    # Мета-тег description
    meta_desc = soup.find('meta', attrs={'name': 'description'})
    if meta_desc and meta_desc.get('content'):
        return meta_desc['content']

    return None

def extract_image(soup):
    # Ищем изображение в мета-тегах
    meta_img = soup.find('meta', property='og:image')
    if meta_img and meta_img.get('content'):
        return meta_img['content']

    # Ищем изображение в JSON-LD
    script = soup.find('script', type='application/ld+json')
    if script:
        try:
            data = json.loads(script.string)
            if isinstance(data, dict) and 'image' in data:
                if isinstance(data['image'], dict) and 'url' in data['image']:
                    return data['image']['url']
                elif isinstance(data['image'], str):
                    return data['image']
        except:
            pass

    # Ищем основное изображение статьи
    img_container = soup.find('div', class_='image') or soup.find('figure')
    if img_container:
        img = img_container.find('img')
        if img and img.get('src'):
            src = img.get('data-src') or img.get('src')
            if src and not src.endswith('.svg'):
                return urljoin(BASE_URL, src)

    # Ищем любое подходящее изображение
    img = soup.find('img', src=True)
    if img and img.get('src'):
        src = img.get('data-src') or img.get('src')
        if src and not any(x in src.lower() for x in ['logo', 'icon', 'avatar', 'banner', 'placeholder']):
            return urljoin(BASE_URL, src)

    return None

def extract_fields(html, url):
    if not html:
        return None

    soup = BeautifulSoup(html, "html.parser")

    # Удаляем служебные теги, но сохраняем JSON-LD
    for tag in soup.find_all(['script', 'style', 'noscript', 'iframe']):
        if tag.get('type') != 'application/ld+json':
            tag.decompose()

    # Извлекаем заголовок
    title = None
    title_elem = soup.find('h1')
    if title_elem:
        title = clean_text(title_elem.get_text())
    if not title and soup.title:
        title = clean_text(soup.title.string)

    # Извлекаем лид
    lead = extract_lead(soup)

    # Извлекаем остальные поля
    content = extract_content(soup)
    excerpt = (content[:EXCERPT_LEN] + '...') if len(content) > EXCERPT_LEN else content
    date = extract_date(soup)
    author = extract_author(soup)
    category = extract_category(soup, url)
    image = extract_image(soup)

    # Создаем хеш на основе контента
    hash_hex = hashlib.sha256((content or "").encode('utf-8')).hexdigest()[:32]

    # Определяем язык (татарский по умолчанию)
    language = "tt"
    if 'rus' in url or 'ru' in url.lower():
        language = "ru"

    return {
        "url": url,
        "title": title or "Без названия",
        "content": content,
        "excerpt": excerpt,
        "lead": lead,
        "description": lead,
        "image_url": image,
        "date": date or datetime.now().isoformat(),
        "category": category,
        "author": author,
        "site": get_domain(url),
        "hash": hash_hex,
        "language": language,
        "scraped_at": datetime.now().isoformat()
    }

# =====================================================
# 🚀 Многопоточное скачивание
# =====================================================
def scrape_articles(article_urls, output_file=OUTPUT_FILE):
    seen_hashes = set()
    seen_urls = set()

    # Загружаем существующие записи
    if os.path.exists(output_file):
        with open(output_file, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    data = json.loads(line.strip())
                    if data.get('hash'):
                        seen_hashes.add(data['hash'])
                    if data.get('url'):
                        seen_urls.add(data['url'])
                except:
                    continue
        print(f"📁 Загружено существующих записей: {len(seen_hashes)}")

    urls_to_process = [u for u in article_urls if u not in seen_urls]
    print(f"🎯 Новых URL для обработки: {len(urls_to_process)}")

    if not urls_to_process:
        print("✅ Все статьи уже обработаны")
        return 0, 0

    successful = 0
    failed = 0

    def worker(url):
        nonlocal successful, failed
        try:
            html = fetch_requests(url)
            if not html:
                failed += 1
                return
            data = extract_fields(html, url)
            if data and data.get('content') and len(data['content']) > 100:
                if data['hash'] not in seen_hashes:
                    seen_hashes.add(data['hash'])
                    with open(output_file, "a", encoding="utf-8") as f:
                        f.write(json.dumps(data, ensure_ascii=False) + "\n")
                    successful += 1
                else:
                    failed += 1
            else:
                failed += 1
        except Exception as e:
            failed += 1
            print(f"❌ Ошибка: {url[:60]}... - {e}")

    with ThreadPoolExecutor(max_workers=MAX_THREADS) as executor:
        futures = {executor.submit(worker, url): url for url in urls_to_process}
        for f in tqdm(as_completed(futures), total=len(futures), desc="Загрузка статей"):
            f.result()

    print(f"✅ Успешно: {successful}, ❌ Ошибок: {failed}")
    print(f"💾 Всего в файле: {len(seen_hashes)}")
    return successful, failed

# =====================================================
# 📊 Анализ результатов
# =====================================================
def analyze_results(filename=OUTPUT_FILE):
    if not os.path.exists(filename):
        print(f"❌ Файл {filename} не найден")
        return

    categories = {}
    authors = {}
    languages = {}
    total_chars = 0
    total_articles = 0

    with open(filename, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                data = json.loads(line.strip())
                total_articles += 1
                cat = data.get('category', 'Не указано')
                categories[cat] = categories.get(cat, 0) + 1
                author = data.get('author', 'Не указан')
                authors[author] = authors.get(author, 0) + 1
                lang = data.get('language', 'tt')
                languages[lang] = languages.get(lang, 0) + 1
                total_chars += len(data.get('content', ''))
            except:
                continue

    print("\n" + "="*60)
    print("📊 СТАТИСТИКА")
    print("="*60)
    print(f"Всего статей: {total_articles}")
    print(f"Всего символов: {total_chars:,}")
    if total_articles > 0:
        print(f"Средняя длина: {total_chars//total_articles} символов")

    print("\n📈 Распределение по языкам:")
    for lang, count in languages.items():
        lang_name = "Татарский" if lang == "tt" else "Русский"
        print(f"  • {lang_name}: {count} ({count/total_articles*100:.1f}%)")

    print("\n📈 Распределение по категориям:")
    for cat, count in sorted(categories.items(), key=lambda x: x[1], reverse=True)[:10]:
        print(f"  • {cat}: {count} ({count/total_articles*100:.1f}%)")

    print("\n✍ Топ авторов:")
    for author, count in sorted(authors.items(), key=lambda x: x[1], reverse=True)[:5]:
        if author != 'Не указан':
            print(f"  • {author}: {count}")

# =====================================================
# 📄 Основной запуск (с возможностью прерывания и ограничениями)
# =====================================================
def main(max_articles=None, max_rubrics=None):
    print("="*70)
    print("🌐 ПАРСЕР СТАТЕЙ КЫЗЫЛ ТАҢ")
    print("="*70)

    all_rubrics = collect_rubric_links()
    if not all_rubrics:
        print("❌ Не найдено рубрик")
        return

    # Если указано максимальное количество рубрик
    if max_rubrics:
        all_rubrics = all_rubrics[:max_rubrics]
        print(f"\n📋 Обрабатываем первые {max_rubrics} рубрик")
    else:
        print(f"\n📋 Всего рубрик для парсинга: {len(all_rubrics)}")

    all_articles = set()

    try:
        for rubric in tqdm(all_rubrics, desc="Сбор статей по рубрикам"):
            urls = collect_articles_from_rubric(rubric)
            all_articles.update(urls)
            time.sleep(random.uniform(*RUBRIC_DELAY))

            # Небольшая пауза между рубриками
            time.sleep(2)

    except KeyboardInterrupt:
        print("\n\n⏸ Прерывание пользователя. Сохраняем собранное...")

    all_articles_list = list(all_articles)
    print(f"\n📊 Всего уникальных статей найдено: {len(all_articles_list)}")

    if max_articles:
        all_articles_list = all_articles_list[:max_articles]
        print(f"🔍 Обрабатываем первых {max_articles} статей")

    if all_articles_list:
        scrape_articles(all_articles_list)
        analyze_results()
        print(f"\n✅ Готово! Файл: {OUTPUT_FILE}")
        print(f"📁 Путь: {os.path.abspath(OUTPUT_FILE)}")
    else:
        print("❌ Нет статей для обработки")

# =====================================================
# 🚀 Запуск
# =====================================================
if __name__ == "__main__":
    try:
        from google.colab import files
        IN_COLAB = True
    except:
        IN_COLAB = False

    # Для теста можно ограничить количество рубрик
    # main(max_articles=None, max_rubrics=5)  # Раскомментировать для теста с 5 рубриками

    # Полный запуск
    main(max_articles=None, max_rubrics=None)

    if IN_COLAB and os.path.exists(OUTPUT_FILE):
        print("\n📥 Скачивание файла...")
        files.download(OUTPUT_FILE)